# A GraphSAGE Approach for Fraud Detection on the IEEE-CIS Dataset
## A Comparative Study with XGBoost on Accuracy, Inference Latency, and Financial Utility

**Research Question:** *Does an inductive GraphSAGE model exploiting shared-entity edges between transactions achieve materially higher AUC and financial savings than an XGBoost tabular baseline on the IEEE-CIS dataset, and at what inference-latency cost per 1,000 transactions?*

---
### Academic Pipeline Workflow
1. **Configuration & Reproducibility:** Global seeds, mode isolation (prototype vs full), hardware profiling.
2. **Data Ingestion & Preprocessing:** Strict chronological 80/10/10 split, memory downcasting, leak-free scaling.
3. **Causal Graph Construction (Table 2 & Fig 1-2):** Anti-supernode degree capping ($k \le 20$), causal enforcement ($t_{src} \le t_{dst}$).
4. **Model Training & Loss Trajectories (Fig 4):** XGBoost, Random Forest, Logistic Regression, GraphSAGE, GAT, and Hybrid.
5. **Benchmarking & Latency Profiling (Table 6 & Fig 3, 5):** Batch 1k and single-stream latency, sampling vs forward decomposition.
6. **Cold-Start & Topological Robustness (Table 7 & Fig 6):** $V_{isolated}$ (deg=0) vs $V_{connected}$ (deg>=3).
7. **Cost-Sensitive Financial Savings (Table 8 & Fig 7-8):** Saved fraud dollars, FP alert costs, net economic savings, parameter sweeps.
8. **Explainability & Tabular Dominance (Fig 9):** XGBoost Gain-based feature attribution.
9. **Model Persistence:** Native JSON/joblib/pt checkpoints and ONNX binary graph export.

## 1. Setup & Environment Configuration

In [ ]:
import os
import sys
import time
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("."))

from src.config import CONFIG, seed_everything, get_device, configure_paths_for_mode
from src.preprocessing import load_and_preprocess
from src.graph_builder import build_causal_transaction_graph, get_graph_statistics
from src.models import train_tabular_baselines, train_graphsage, train_gat
from src.benchmark import (
    compute_classification_metrics,
    predict_gnn_probabilities,
    benchmark_tabular_latency,
    benchmark_gnn_latency,
    evaluate_cold_start_robustness,
    compute_financial_savings,
    sweep_cost_sensitivity,
    export_results_table,
    export_latency_table,
    export_overall_metrics_table,
    export_training_times_table,
)
from src.visualization import (
    plot_degree_distribution,
    plot_temporal_fraud_distribution,
    plot_pr_and_roc_curves,
    plot_accuracy_latency_tradeoff,
    plot_latency_breakdown,
    plot_cold_start_comparison,
    plot_graphsage_training_curves,
    plot_financial_savings_decomposition,
    plot_cost_sensitivity,
    plot_feature_importance,
    create_paper_figure_aliases,
)
from src.serialization import save_native_models, export_tabular_to_onnx, train_hybrid_model

# Configure execution mode: "prototype" (100k tx) or "full" (590k tx)
config = configure_paths_for_mode(CONFIG, mode="prototype")
seed_everything(config["seed"])
device = get_device()
print(f"Active Device: {device} | Output Directory: {config['results_dir']}")

## 2. Leak-Free Preprocessing & Chronological Splitting

In [ ]:
data_dict = load_and_preprocess(config)
plot_temporal_fraud_distribution(data_dict, config)
plt.show()

## 3. Causal Graph Construction (Table 2 & Anti-Supernode Pruning)

In [ ]:
graph_data = build_causal_transaction_graph(data_dict, config)
display(graph_data.edge_stats_df)
plot_degree_distribution(graph_data, config)
plt.show()

## 4. Train Tabular Baselines (XGBoost, RF, LR)

In [ ]:
tabular_models, tabular_times = train_tabular_baselines(data_dict, config)
print("Tabular training times:", tabular_times)

## 5. Train Inductive GraphSAGE & Training Curves (Fig 4)

In [ ]:
graphsage_model, gs_history, t_graphsage = train_graphsage(graph_data, config, device)
plot_graphsage_training_curves(gs_history, config)
plt.show()

## 6. Train Appendix Baseline (GAT) & Hybrid Model (GraphSAGE + XGBoost)

In [ ]:
gat_model, t_gat = train_gat(graph_data, config, device)
hybrid_model, X_hybrid_test, t_hybrid = train_hybrid_model(graphsage_model, graph_data, data_dict, config, device)

training_times = {**tabular_times, "GraphSAGE": t_graphsage, "GAT": t_gat, "Hybrid": t_hybrid}
df_times = export_training_times_table(training_times, config)
display(df_times)

## 7. Out-of-Sample Performance Evaluation (Table 1 & PR/ROC Curves)

In [ ]:
test_idx = data_dict["test_idx"]
y_test = data_dict["y"][test_idx]
X_tab_test = data_dict["X_tabular"][test_idx]
X_neu_test = data_dict["X_neural"][test_idx]

predictions_dict = {}
predictions_dict["XGBoost"] = tabular_models["XGBoost"].predict_proba(X_tab_test)[:, 1]
predictions_dict["RandomForest"] = tabular_models["RandomForest"].predict_proba(X_tab_test)[:, 1]
predictions_dict["LogisticRegression"] = tabular_models["LogisticRegression"].predict_proba(X_neu_test)[:, 1]

_, gs_prob = predict_gnn_probabilities(graphsage_model, graph_data, graph_data.test_mask, config, device)
predictions_dict["GraphSAGE"] = gs_prob

_, gat_prob = predict_gnn_probabilities(gat_model, graph_data, graph_data.test_mask, config, device)
predictions_dict["GAT"] = gat_prob

predictions_dict["Hybrid"] = hybrid_model.predict_proba(X_hybrid_test)[:, 1]

metrics_dict = {m: compute_classification_metrics(y_test, p) for m, p in predictions_dict.items()}
table1_df = pd.DataFrame([{"Model": m, **v} for m, v in metrics_dict.items()])
display(table1_df)

export_results_table(table1_df, "table1_classification_metrics", config, caption="Test Set Metrics")
export_overall_metrics_table(metrics_dict, training_times, None, config)
plot_pr_and_roc_curves(predictions_dict, y_test, config)
plt.show()

## 8. Dual Inference Latency Benchmarking (Table 6 & Pareto Frontier)

In [ ]:
sample_1k_tab = X_tab_test[:1000]
sample_1k_neu = X_neu_test[:1000]

tabular_latency_bench = {
    "XGBoost": dict(zip(["batch_mean", "batch_std"], benchmark_tabular_latency(tabular_models["XGBoost"], sample_1k_tab))),
    "RandomForest": dict(zip(["batch_mean", "batch_std"], benchmark_tabular_latency(tabular_models["RandomForest"], sample_1k_tab))),
    "LogisticRegression": dict(zip(["batch_mean", "batch_std"], benchmark_tabular_latency(tabular_models["LogisticRegression"], sample_1k_neu))),
}
for m in ["XGBoost", "RandomForest", "LogisticRegression"]:
    s_mean, s_std = benchmark_tabular_latency(tabular_models[m], sample_1k_neu[:1] if m == "LogisticRegression" else sample_1k_tab[:1])
    tabular_latency_bench[m]["single_mean"] = s_mean
    tabular_latency_bench[m]["single_std"] = s_std

gnn_1k = {
    "GraphSAGE": benchmark_gnn_latency(graphsage_model, graph_data, test_idx, 1000, config, device),
    "GAT": benchmark_gnn_latency(gat_model, graph_data, test_idx, 1000, config, device),
}
gnn_1tx = {
    "GraphSAGE": benchmark_gnn_latency(graphsage_model, graph_data, test_idx, 1, config, device),
    "GAT": benchmark_gnn_latency(gat_model, graph_data, test_idx, 1, config, device),
}

df_table6 = export_latency_table(tabular_latency_bench, gnn_1k, gnn_1tx, config)
display(df_table6)

plot_latency_breakdown(gnn_1k, config)
plt.show()

## 9. Cold-Start Robustness Analysis (Table 7 & Fig 6)

In [ ]:
df_robustness = evaluate_cold_start_robustness(tabular_models, graphsage_model, data_dict, graph_data, config, device)
display(df_robustness)
plot_cold_start_comparison(df_robustness, config)
plt.show()

## 10. Cost-Sensitive Financial Savings Analysis (Table 8 & Fig 7-8)

In [ ]:
transaction_amt_test = data_dict["transaction_amt"][test_idx]
df_financial = compute_financial_savings(y_test, predictions_dict, transaction_amt_test, config, c_fp=5.0)
display(df_financial)
plot_financial_savings_decomposition(df_financial, config)
plt.show()

df_sweep = sweep_cost_sensitivity(y_test, predictions_dict, transaction_amt_test, config)
plot_cost_sensitivity(df_sweep, config)
plt.show()

## 11. Explainability Analysis (Fig 9: XGBoost Feature Importance)

In [ ]:
plot_feature_importance(tabular_models["XGBoost"], data_dict["feature_names"], config, top_k=20)
plt.show()

## 12. Model Checkpointing & Paper Figure Aliasing

In [ ]:
save_native_models(tabular_models, graphsage_model, gat_model, config, hybrid_model=hybrid_model)
export_tabular_to_onnx(tabular_models, n_features=data_dict["X_tabular"].shape[1], config=config)
create_paper_figure_aliases(config)
print("All models and paper figures finalized successfully!")